# Modelo para escolha de plano telefônico

## Descrição do projeto

A operadora de celular Megaline está insatisfeita com o fato de muitos de seus clientes estarem usando planos antigos. Ela quer desenvolver um modelo que possa analisar o comportamento do cliente e recomendar um dos planos mais recentes da Megaline: Smart ou Ultra.

Você tem acesso a dados de comportamento dos clientes que já mudaram para os novos planos (do projeto do curso de [Análise de dados estatísticos](https://github.com/peperrone/mbads-sprint5-AED)). Para essa tarefa de classificação, você precisa desenvolver um modelo que escolhe o plano certo. Como você já executou a etapa de pré-processamento de dados, pode ir direto para a criação do modelo.

Desenvolva um modelo com a maior acurácia possível. Neste projeto, o limite para acurácia é 0,75. Verifique a acurácia usando o conjunto de dados de teste.

### Instruções do projeto
- <input type="checkbox" checked> Abra e examine o arquivo de dados.
- <input type="checkbox" checked> Divida os dados de origem em um conjunto de treinamento, um conjunto de validação e um conjunto de teste.
- <input type="checkbox" checked> Investigue a qualidade de diferentes modelos alterando hiperparâmetros. Descreva brevemente os resultados do estudo.
- <input type="checkbox" checked> Verifique a qualidade do modelo usando o conjunto de teste.

**Tarefa adicional:** 
- <input type="checkbox"> Tirar a prova real do modelo. Esses dados são mais complexos do que os que você está acostumado a trabalhar, então não será uma tarefa fácil. Vamos dar uma olhada mais de perto mais tarde

## Inicialização

In [91]:
#importando bibliotecas
import numpy as np
import pandas as pd

In [92]:
# Importando o dataset
try:
    df = pd.read_csv('datasets/users_behavior.csv') # caminho local no windows
except FileNotFoundError:
    df = pd.read_csv('/datasets/users_behavior.csv') # caminho no ambiente da tripleten

## Examinando os dados

In [93]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [94]:
df.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [95]:
df.describe()

,calls,minutes,messages,mb_used,is_ultra
count,3214.000000,3214.000000,3214.000000,3214.000000,3214.000000
mean,63.038892,438.208787,38.281269,17207.673836,0.306472
std,33.236368,234.569872,36.148326,7570.968246,0.461100
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,40.000000,274.575000,9.000000,12491.902500,0.000000
50%,62.000000,430.600000,30.000000,16943.235000,0.000000
75%,82.000000,571.927500,57.000000,21424.700000,1.000000
max,244.000000,1632.060000,224.000000,49745.730000,1.000000


In [96]:
# Verificando a distribuição da variável alvo
print(df['is_ultra'].value_counts())

is_ultra
0    2229
1     985
Name: count, dtype: int64


In [ ]:
# Dividindo o dataset nas nossas duas categorias: ultra e smart (não ultra)
df_ultra = df[df['is_ultra'] == 1]
df_smart = df[df['is_ultra'] == 0]

In [98]:
df_ultra.describe()

,calls,minutes,messages,mb_used,is_ultra
count,985.000000,985.000000,985.000000,985.000000,985.0
mean,73.392893,511.224569,49.363452,19468.823228,1.0
std,43.916853,308.031100,47.804457,10087.178654,0.0
min,0.000000,0.000000,0.000000,0.000000,1.0
25%,41.000000,276.030000,6.000000,11770.280000,1.0
50%,74.000000,502.550000,38.000000,19308.010000,1.0
75%,104.000000,730.050000,79.000000,26837.720000,1.0
max,244.000000,1632.060000,224.000000,49745.730000,1.0


In [ ]:
df_smart.describe()

,calls,minutes,messages,mb_used,is_ultra
count,2229.000000,2229.000000,2229.000000,2229.000000,2229.0
mean,58.463437,405.942952,33.384029,16208.466949,0.0
std,25.939858,184.512604,28.227876,5870.498853,0.0
min,0.000000,0.000000,0.000000,0.000000,0.0
25%,40.000000,274.230000,10.000000,12643.050000,0.0
50%,60.000000,410.560000,28.000000,16506.930000,0.0
75%,76.000000,529.510000,51.000000,20043.060000,0.0
max,198.000000,1390.220000,143.000000,38552.620000,0.0


## Preparação dos conjuntos de treinamento, validação e teste

In [100]:
# Importando a função para dividir o dataset
from sklearn.model_selection import train_test_split

In [101]:
# Dividindo o dataset em target e features
features = df.drop('is_ultra', axis=1) 
target = df['is_ultra']

# Dividindo o dataset em treino e o subconjunto de teste+validação
features_train, features_test_val, target_train, target_test_val = train_test_split(features, target, test_size=0.4, random_state=42)

In [102]:
## Dividindo o subcojunto em teste e validação
# 50% do subconjunto de teste+validação para cada um (20% do total para cada um)
features_val, features_test, target_val, target_test = train_test_split(features_test_val, target_test_val, test_size=0.5, random_state=42) 

## Testando modelos

In [103]:
# Importando modelos de classificação
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Importando as funções para avaliar os modelos
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### Regressão logística

In [104]:
# Criando uma instância do modelo de regressão logística
log_model = LogisticRegression(random_state=42, solver='liblinear') # solver 'liblinear' é recomendado para conjuntos de dados pequenos

# Treinando o modelo de regressão logística
log_model.fit(features_train, target_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [105]:
# Valiando o modelo de regressão logística
log_predictions = log_model.predict(features_val)

log_accuracy = accuracy_score(target_val, log_predictions)
print(f"Acurácia do modelo de Regressão Logística: {log_accuracy:.2f}")
print("\nRelatório de Classificação:\n", classification_report(target_val, log_predictions))
print("\nMatriz de Confusão:\n", confusion_matrix(target_val, log_predictions))

Acurácia do modelo de Regressão Logística: 0.72

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.72      0.98      0.83       446
           1       0.79      0.14      0.23       197

    accuracy                           0.72       643
   macro avg       0.76      0.56      0.53       643
weighted avg       0.74      0.72      0.65       643


Matriz de Confusão:
 [[439   7]
 [170  27]]


In [106]:
# Fazendo testes com outros solvers do modelo de regressão logística
solvers = ['lbfgs', 'liblinear', 'newton-cg'] # excluindo sag e saga por serem mais lentos em conjuntos de dados pequenos e liblinear por já ter sido testado
for solver in solvers:
    log_model = LogisticRegression(random_state=42, solver=solver)
    log_model.fit(features_train, target_train)
    predictions = log_model.predict(features_val)
    accuracy = accuracy_score(target_val, predictions)
    print(f"Solver: {solver}, Acurácia: {accuracy:.2f}")
    print("\nRelatório de Classificação:\n", classification_report(target_val, predictions))
    print("\nMatriz de Confusão:\n", confusion_matrix(target_val, predictions))
    print()



Solver: lbfgs, Acurácia: 0.74

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.74      0.97      0.84       446
           1       0.78      0.21      0.33       197

    accuracy                           0.74       643
   macro avg       0.76      0.59      0.59       643
weighted avg       0.75      0.74      0.68       643


Matriz de Confusão:
 [[434  12]
 [155  42]]

Solver: liblinear, Acurácia: 0.72

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.72      0.98      0.83       446
           1       0.79      0.14      0.23       197

    accuracy                           0.72       643
   macro avg       0.76      0.56      0.53       643
weighted avg       0.74      0.72      0.65       643


Matriz de Confusão:
 [[439   7]
 [170  27]]

Solver: newton-cg, Acurácia: 0.74

Relatório de Classificação:
               precision    recall  f1-score   support

           

In [107]:
# Seguindo com lbfgs, que teve a melhor acurácia e testando hiperparâmetros do modelo de regressão logística
log_model = LogisticRegression(random_state=42, solver='lbfgs', class_weight='balanced') # class_weight='balanced' é recomendado para lidar com classes desbalanceadas
log_model.fit(features_train, target_train)
log_predictions = log_model.predict(features_val)
log_accuracy = accuracy_score(target_val, log_predictions)
print(f"Acurácia do modelo de Regressão Logística com class_weight='balanced': {log_accuracy:.2f}")
print("\nRelatório de Classificação:\n", classification_report(target_val, log_predictions))
print("\nMatriz de Confusão:\n", confusion_matrix(target_val, log_predictions))

Acurácia do modelo de Regressão Logística com class_weight='balanced': 0.63

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.78      0.64      0.71       446
           1       0.42      0.60      0.50       197

    accuracy                           0.63       643
   macro avg       0.60      0.62      0.60       643
weighted avg       0.67      0.63      0.64       643


Matriz de Confusão:
 [[286 160]
 [ 79 118]]


### Árvore de decisão

In [108]:
# Criando uma instância do modelo de árvore de decisão
tree_model = DecisionTreeClassifier(random_state=42)
# Treinando o modelo de árvore de decisão
tree_model.fit(features_train, target_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [109]:
# Valiando o modelo de árvore de decisão
tree_predictions = tree_model.predict(features_val)

tree_accuracy = accuracy_score(target_val, tree_predictions)
print(f"Acurácia do modelo de Árvore de Decisão: {tree_accuracy:.2f}")
print("\nRelatório de Classificação:\n", classification_report(target_val, tree_predictions))
print("\nMatriz de Confusão:\n", confusion_matrix(target_val, tree_predictions))

Acurácia do modelo de Árvore de Decisão: 0.73

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.80      0.80      0.80       446
           1       0.56      0.56      0.56       197

    accuracy                           0.73       643
   macro avg       0.68      0.68      0.68       643
weighted avg       0.73      0.73      0.73       643


Matriz de Confusão:
 [[358  88]
 [ 87 110]]


In [110]:
# Criando um loop para testar diferentes hiperparâmetros do modelo de árvore de decisão
best_depth = None
for depth in range(1, 21): # testando profundidades de 1 a 10
    tree_model = DecisionTreeClassifier(random_state=42, max_depth=depth)
    tree_model.fit(features_train, target_train)
    predictions = tree_model.predict(features_val)
    accuracy = accuracy_score(target_val, predictions)
    print(f"Profundidade: {depth}, Acurácia: {accuracy:.2f}")
    print("\nRelatório de Classificação:\n", classification_report(target_val, predictions))
    print("\nMatriz de Confusão:\n", confusion_matrix(target_val, predictions))
    print()
    if accuracy > tree_accuracy:
        tree_accuracy = accuracy
        best_depth = depth

Profundidade: 1, Acurácia: 0.73

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.73      0.97      0.83       446
           1       0.75      0.18      0.29       197

    accuracy                           0.73       643
   macro avg       0.74      0.58      0.56       643
weighted avg       0.74      0.73      0.67       643


Matriz de Confusão:
 [[434  12]
 [161  36]]

Profundidade: 2, Acurácia: 0.78

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.78      0.95      0.86       446
           1       0.79      0.40      0.53       197

    accuracy                           0.78       643
   macro avg       0.78      0.67      0.69       643
weighted avg       0.78      0.78      0.76       643




Matriz de Confusão:
 [[425  21]
 [119  78]]

Profundidade: 3, Acurácia: 0.79

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.79      0.95      0.86       446
           1       0.78      0.44      0.56       197

    accuracy                           0.79       643
   macro avg       0.79      0.69      0.71       643
weighted avg       0.79      0.79      0.77       643


Matriz de Confusão:
 [[422  24]
 [110  87]]

Profundidade: 4, Acurácia: 0.78

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.79      0.94      0.86       446
           1       0.76      0.42      0.54       197

    accuracy                           0.78       643
   macro avg       0.77      0.68      0.70       643
weighted avg       0.78      0.78      0.76       643


Matriz de Confusão:
 [[420  26]
 [115  82]]

Profundidade: 5, Acurácia: 0.77

Relatório de Classificação:
               precisio

In [111]:
# Imprimindo a melhor profundidade encontrada e a acurácia correspondente
print(f"Melhor profundidade: {best_depth} com acurácia de {tree_accuracy:.2f}")

Melhor profundidade: 8 com acurácia de 0.80


In [112]:
# Criando uma instância do modelo de árvore de decisão com class_weight='balanced'
tree_model_balanced = DecisionTreeClassifier(random_state=42, max_depth=best_depth, class_weight='balanced')
tree_model_balanced.fit(features_train, target_train)
tree_predictions_balanced = tree_model_balanced.predict(features_val)
tree_accuracy = accuracy_score(target_val, tree_predictions_balanced)
print(f"Acurácia do modelo de Árvore de Decisão: {tree_accuracy:.2f}")
print("\nRelatório de Classificação:\n", classification_report(target_val, tree_predictions_balanced))
print("\nMatriz de Confusão:\n", confusion_matrix(target_val, tree_predictions_balanced))


Acurácia do modelo de Árvore de Decisão: 0.73

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.80      0.83      0.81       446
           1       0.57      0.52      0.55       197

    accuracy                           0.73       643
   macro avg       0.68      0.68      0.68       643
weighted avg       0.73      0.73      0.73       643


Matriz de Confusão:
 [[369  77]
 [ 94 103]]


### Floresta aleatória

In [113]:
# Criando uma instância do modelo de floresta aleatória
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(features_train, target_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [114]:
rf_predictions = rf_model.predict(features_val)
rf_accuracy = accuracy_score(target_val, rf_predictions)
print(f"Acurácia do modelo de Floresta Aleatória: {rf_accuracy:.2f}")
print("\nRelatório de Classificação:\n", classification_report(target_val, rf_predictions))
print("\nMatriz de Confusão:\n", confusion_matrix(target_val, rf_predictions)) 

Acurácia do modelo de Floresta Aleatória: 0.80

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.82      0.92      0.87       446
           1       0.75      0.54      0.63       197

    accuracy                           0.80       643
   macro avg       0.78      0.73      0.75       643
weighted avg       0.80      0.80      0.79       643


Matriz de Confusão:
 [[410  36]
 [ 91 106]]


In [115]:
# Criando um loop para testar diferentes hiperparâmetros do modelo de árvore de decisão
for n in range(1, 100): # testando profundidades de 1 a 20
    rf_model = RandomForestClassifier(random_state=42, n_estimators=n)
    rf_model.fit(features_train, target_train)
    predictions = rf_model.predict(features_val)
    accuracy = accuracy_score(target_val, predictions)
    print(f"Profundidade: {n}, Acurácia: {accuracy:.2f}")
    print("\nRelatório de Classificação:\n", classification_report(target_val, predictions))
    print("\nMatriz de Confusão:\n", confusion_matrix(target_val, predictions))
    print()
    if accuracy > rf_accuracy:
        rf_accuracy = accuracy
        best_n = n

Profundidade: 1, Acurácia: 0.72

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.80      0.79      0.79       446
           1       0.53      0.55      0.54       197

    accuracy                           0.72       643
   macro avg       0.67      0.67      0.67       643
weighted avg       0.72      0.72      0.72       643


Matriz de Confusão:
 [[351  95]
 [ 88 109]]

Profundidade: 2, Acurácia: 0.77

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.78      0.93      0.85       446
           1       0.71      0.42      0.53       197

    accuracy                           0.77       643
   macro avg       0.75      0.67      0.69       643
weighted avg       0.76      0.77      0.75       643


Matriz de Confusão:
 [[413  33]
 [115  82]]

Profundidade: 3, Acurácia: 0.76

Relatório de Classificação:
               precision    recall  f1-score   support

           0 

In [116]:
print(f"Melhor número de árvores: {best_n} com acurácia de {rf_accuracy:.2f}")

Melhor número de árvores: 91 com acurácia de 0.81


In [117]:
# Testando o modelo de floresta aleatória com class_weight='balanced'
rf_model_balanced = RandomForestClassifier(random_state=42, n_estimators=best_n, class_weight='balanced')
rf_model_balanced.fit(features_train, target_train)
rf_predictions_balanced = rf_model_balanced.predict(features_val)
rf_accuracy_balanced = accuracy_score(target_val, rf_predictions_balanced)
print(f"Acurácia do modelo de Floresta Aleatória com class_weight='balanced': {rf_accuracy_balanced:.2f}")
print("\nRelatório de Classificação:\n", classification_report(target_val, rf_predictions_balanced))
print("\nMatriz de Confusão:\n", confusion_matrix(target_val, rf_predictions_balanced))

Acurácia do modelo de Floresta Aleatória com class_weight='balanced': 0.80

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.82      0.92      0.86       446
           1       0.74      0.54      0.62       197

    accuracy                           0.80       643
   macro avg       0.78      0.73      0.74       643
weighted avg       0.79      0.80      0.79       643


Matriz de Confusão:
 [[409  37]
 [ 91 106]]


## Testes de hiperparâmetros com GridSearch

In [118]:
# Importando GridSearchCV
from sklearn.model_selection import GridSearchCV

### Regressão logística

In [119]:
# Definindo o grid de hiperparâmetros para o modelo de regressão logística
param_grid_log = {
    'solver': ['lbfgs', 'liblinear', 'newton-cg'],
    'class_weight': ['balanced', None],
    'C': [0.01, 0.1, 1, 10], # C é o inverso da regularização, valores menores significam mais regularização
    'max_iter': [100, 200, 1000, 5000], # número máximo de iterações para o algoritmo de otimização
}

In [120]:
# Reiniciando o modelo de regressão logística para usar no GridSearchCV
log_model = LogisticRegression(random_state=42)

# Criando o GridSearchCV para o modelo de regressão logística
gs_log_model = GridSearchCV(log_model, param_grid=param_grid_log, cv=5, scoring='accuracy') # refit='accuracy' para escolher o melhor modelo com base na acurácia

In [121]:
gs_log_model.fit(features_train, target_train)

e:\DataScience\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
e:\DataScience\anaconda3\Lib\site-packages\scipy\optimize\_linesearch.py:312: LineSearchWarning: The line search algorithm did not converge
  alpha_star, phi_star, old_fval, derphi_star = scalar_search_wolfe2(
e:\DataScience\anaconda3\Lib\site-packages\sklearn\utils\optimize.py:100: LineSearchWarning: The line search algorithm did not converge
  ret = line_search_wolfe2(
e:\DataScience\anaconda3\Lib\site-pack

,estimator,LogisticRegre...ndom_state=42)
,param_grid,"{'C': [0.01, 0.1, ...], 'class_weight': ['balanced', None], 'max_iter': [100, 200, ...], 'solver': ['lbfgs', 'liblinear', ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [122]:
print(f"Melhores hiperparâmetros encontrados: {gs_log_model.best_params_}")
print(f"Melhor acurácia encontrada: {gs_log_model.best_score_:.2f}")

Melhores hiperparâmetros encontrados: {'C': 0.01, 'class_weight': None, 'max_iter': 100, 'solver': 'lbfgs'}
Melhor acurácia encontrada: 0.74


### Árvore de decisão

In [123]:
# Definindo o grid de hiperparâmetros para o modelo de árvore de decisão
param_grid_tree = {
    'criterion': ['gini', 'entropy', 'log_loss'], 
    'max_depth': range(1, 21), # testando profundidades de 1 a 20
    'class_weight': ['balanced', None],
    'splitter': ['best', 'random'] # testando os dois tipos de divisão
}

In [124]:
gs_tree_model = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid=param_grid_tree, cv=5, scoring='accuracy')
gs_tree_model.fit(features_train, target_train)

,estimator,DecisionTreeC...ndom_state=42)
,param_grid,"{'class_weight': ['balanced', None], 'criterion': ['gini', 'entropy', ...], 'max_depth': range(1, 21), 'splitter': ['best', 'random']}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [125]:
print(f"Melhores hiperparâmetros encontrados: {gs_tree_model.best_params_}")
print(f"Melhor acurácia encontrada: {gs_tree_model.best_score_:.2f}")

Melhores hiperparâmetros encontrados: {'class_weight': None, 'criterion': 'gini', 'max_depth': 6, 'splitter': 'random'}
Melhor acurácia encontrada: 0.80


### Floresta aleatória

In [126]:
# Definindo o grid de hiperparâmetros para o modelo de floresta aleatória
param_grid_rf = {
    'n_estimators': [10, 50, 100, 200], # testando diferentes números de árvores
    'max_depth': [None, 10, 20, 30], # testando diferentes profundidades máximas
    'class_weight': ['balanced', None], # testando o uso de class_weight para lidar com classes desbalanceadas
    'criterion': ['gini', 'entropy', 'log_loss'] # testando diferentes critérios de divisão
}

In [127]:
# Criando o GridSearchCV para o modelo de floresta aleatória
rf_model = RandomForestClassifier(random_state=42)
gs_rf_model = GridSearchCV(rf_model, param_grid=param_grid_rf, cv=5, scoring='accuracy') # refit='accuracy' para escolher o melhor modelo com base na acurácia

In [128]:
# Treinando o GridSearchCV para o modelo de floresta aleatória
gs_rf_model.fit(features_train, target_train)

,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'class_weight': ['balanced', None], 'criterion': ['gini', 'entropy', ...], 'max_depth': [None, 10, ...], 'n_estimators': [10, 50, ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,50


In [129]:
print(f"Melhores hiperparâmetros encontrados: {gs_rf_model.best_params_}")
print(f"Melhor acurácia encontrada: {gs_rf_model.best_score_:.2f}")

Melhores hiperparâmetros encontrados: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 10, 'n_estimators': 50}
Melhor acurácia encontrada: 0.81


In [130]:
# Treinando o GridSearchCV para o modelo de floresta aleatória usando a métrica F1 para escolher o melhor modelo, priorizando Ultra
gs_rf_model_f1 = GridSearchCV(rf_model, param_grid=param_grid_rf, cv=5, scoring='f1')
gs_rf_model_f1.fit(features_train, target_train)

,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'class_weight': ['balanced', None], 'criterion': ['gini', 'entropy', ...], 'max_depth': [None, 10, ...], 'n_estimators': [10, 50, ...]}"
,scoring,'f1'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [131]:
print(f"Melhores hiperparâmetros encontrados: {gs_rf_model_f1.best_params_}")
print(f"Melhor f1 encontrado: {gs_rf_model_f1.best_score_:.2f}")

Melhores hiperparâmetros encontrados: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 10, 'n_estimators': 200}
Melhor f1 encontrado: 0.63


In [132]:
# Treinando o GridSearchCV para o modelo de floresta aleatória usando a métrica F1 macro para escolher o melhor modelo, tentando balancear o peso das duas classes
gs_rf_model_f1m = GridSearchCV(rf_model, param_grid=param_grid_rf, cv=5, scoring='f1_macro') 
gs_rf_model_f1m.fit(features_train, target_train)

,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'class_weight': ['balanced', None], 'criterion': ['gini', 'entropy', ...], 'max_depth': [None, 10, ...], 'n_estimators': [10, 50, ...]}"
,scoring,'f1_macro'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,50


In [133]:
print(f"Melhores hiperparâmetros encontrados: {gs_rf_model_f1m.best_params_}")
print(f"Melhor f1_macro encontrado: {gs_rf_model_f1m.best_score_:.2f}")

Melhores hiperparâmetros encontrados: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 10, 'n_estimators': 50}
Melhor f1_macro encontrado: 0.75


### Avaliando as métricas dos melhores modelos em mais detalhes

In [134]:
# Salvando os melhores parâmetros encontrados para cada modelo em um dicionário para cada modelo
log_params = gs_log_model.best_params_
tree_params = gs_tree_model.best_params_
rf_params = gs_rf_model.best_params_
rf_f1_params = gs_rf_model_f1.best_params_
rf_f1m_params = gs_rf_model_f1m.best_params_

In [135]:
# Instanciando os modelos com os melhores hiperparâmetros encontrados
best_log_model = LogisticRegression(random_state=42, **log_params)
best_tree_model = DecisionTreeClassifier(random_state=42, **tree_params)
best_rf_model = RandomForestClassifier(random_state=42, **rf_params)

In [136]:
# Treinando e avaliando métricas para o modelo de regressão logística com os melhores hiperparâmetros
best_log_model.fit(features_train, target_train)
log_predictions = best_log_model.predict(features_val)
log_accuracy = accuracy_score(target_val, log_predictions)
print(f"Acurácia do modelo de Regressão Logística com os melhores hiperparâmetros: {log_accuracy:.2f}")
print("\nRelatório de Classificação:\n", classification_report(target_val, log_predictions))
print("\nMatriz de Confusão:\n", confusion_matrix(target_val, log_predictions))

Acurácia do modelo de Regressão Logística com os melhores hiperparâmetros: 0.74



Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.74      0.97      0.84       446
           1       0.78      0.21      0.33       197

    accuracy                           0.74       643
   macro avg       0.76      0.59      0.59       643
weighted avg       0.75      0.74      0.68       643


Matriz de Confusão:
 [[434  12]
 [155  42]]


In [137]:
# Treinando e avaliando o modelo de árvore de decisão com os melhores hiperparâmetros encontrados
best_tree_model.fit(features_train, target_train)
tree_predictions = best_tree_model.predict(features_val)
tree_accuracy = accuracy_score(target_val, tree_predictions)
print(f"Acurácia do modelo de Árvore de Decisão com os melhores hiperparâmetros: {tree_accuracy:.2f}")
print("\nRelatório de Classificação:\n", classification_report(target_val, tree_predictions))
print("\nMatriz de Confusão:\n", confusion_matrix(target_val, tree_predictions))

Acurácia do modelo de Árvore de Decisão com os melhores hiperparâmetros: 0.77

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.79      0.93      0.85       446
           1       0.72      0.43      0.54       197

    accuracy                           0.77       643
   macro avg       0.75      0.68      0.70       643
weighted avg       0.77      0.77      0.76       643


Matriz de Confusão:
 [[413  33]
 [112  85]]


In [138]:
# Treinando e avaliando o modelo de floresta aleatória com os melhores hiperparâmetros encontrados
best_rf_model.fit(features_train, target_train)
rf_predictions = best_rf_model.predict(features_val)
rf_accuracy = accuracy_score(target_val, rf_predictions)
print(f"Acurácia do modelo de Floresta Aleatória com os melhores hiperparâmetros: {rf_accuracy:.2f}")
print("\nRelatório de Classificação:\n", classification_report(target_val, rf_predictions))
print("\nMatriz de Confusão:\n", confusion_matrix(target_val, rf_predictions))

Acurácia do modelo de Floresta Aleatória com os melhores hiperparâmetros: 0.81

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.81      0.95      0.87       446
           1       0.80      0.50      0.61       197

    accuracy                           0.81       643
   macro avg       0.81      0.72      0.74       643
weighted avg       0.81      0.81      0.79       643


Matriz de Confusão:
 [[422  24]
 [ 99  98]]


In [139]:
# Treinando e avaliando o modelo de floresta aleatória otimizado para F1 com os melhores hiperparâmetros encontrados
best_rf_model_f1 = RandomForestClassifier(random_state=42, **rf_f1_params)
best_rf_model_f1.fit(features_train, target_train)
rf_predictions_f1 = best_rf_model_f1.predict(features_val)
rf_accuracy_f1 = accuracy_score(target_val, rf_predictions_f1)
print(f"Acurácia do modelo de Floresta Aleatória otimizado para F1 com os melhores hiperparâmetros: {rf_accuracy_f1:.2f}")
print("\nRelatório de Classificação:\n", classification_report(target_val, rf_predictions_f1))
print("\nMatriz de Confusão:\n", confusion_matrix(target_val, rf_predictions_f1))

Acurácia do modelo de Floresta Aleatória otimizado para F1 com os melhores hiperparâmetros: 0.80

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.82      0.91      0.87       446
           1       0.74      0.55      0.63       197

    accuracy                           0.80       643
   macro avg       0.78      0.73      0.75       643
weighted avg       0.80      0.80      0.79       643


Matriz de Confusão:
 [[407  39]
 [ 88 109]]


In [140]:
# Treinando e avaliando o modelo de floresta aleatória otimizado para F1_macro com os melhores hiperparâmetros encontrados
best_rf_model_f1m = RandomForestClassifier(random_state=42, **rf_f1m_params)
best_rf_model_f1m.fit(features_train, target_train)
rf_predictions_f1m = best_rf_model_f1m.predict(features_val)
rf_accuracy_f1m = accuracy_score(target_val, rf_predictions_f1m)
print(f"Acurácia do modelo de Floresta Aleatória otimizado para F1_macro com os melhores hiperparâmetros: {rf_accuracy_f1m:.2f}")
print("\nRelatório de Classificação:\n", classification_report(target_val, rf_predictions_f1m))
print("\nMatriz de Confusão:\n", confusion_matrix(target_val, rf_predictions_f1m))

Acurácia do modelo de Floresta Aleatória otimizado para F1_macro com os melhores hiperparâmetros: 0.81

Relatório de Classificação:
               precision    recall  f1-score   support

           0       0.81      0.95      0.87       446
           1       0.80      0.50      0.61       197

    accuracy                           0.81       643
   macro avg       0.81      0.72      0.74       643
weighted avg       0.81      0.81      0.79       643


Matriz de Confusão:
 [[422  24]
 [ 99  98]]


In [141]:
chosen_model = best_rf_model_f1

## Observações e escolha do modelo

**Comparação entre os modelos:**

* **Regressão Logística:** apresentou resultados abaixo do desejado em termos de acurácia (cerca de 74% v. mínimo exigido de 75%), mas teve dificuldades para identificar corretamente a classe positiva (Ultra), obtendo baixo recall e F1-score para essa classe. Ao utilizar class_weight='balanced', o recall da classe minoritária melhorou, porém houve uma queda significativa na acurácia geral do modelo.
* **Árvore de Decisão:** apresentou desempenho superior ao da Regressão Logística, alcançando aproximadamente 80% de acurácia com profundidade máxima igual a 8. Entretanto, o modelo mostrou desempenho menos consistente ao tentar equilibrar as classes.
* **Floresta Aleatória:** foi o modelo com melhor desempenho geral. Após testes com diferentes números de árvores e ajuste de hiperparâmetros por meio de Grid Search, o modelo atingiu cerca de 80% de acurácia e apresentou os melhores valores de F1-score para a classe positiva. 

**Dados desbalanceados:**

* Temos dados desbalanceados, 69.3% de planos Smart para 30.7% de planos Ultra. E isso afetou o desempenho do modelo, que funcionou muito melhor para identificar usuários Smart do que Ultra. 
* Fiz alguns testes utilizando ``class-weight='balanced'`` para tentar corrigir, o que afetou a acurácia de todos os modelos. O menor impacto foi observado no modelo `RandomForestClassifier` (RFC), que também obteve a melhor acurácia. 
    * Como RFC foi menos afetado, decidi explorar mais alguns critérios de otimização com GridSearcherCV nessa classe de modelos.

**Escolha do modelo:**

* Para a escolha do modelo, me baseei nas observações do estudo em [Análise Exploratória de Dados](https://github.com/peperrone/mbads-sprint5-AED) feita com dados da Megaline anteriormente, que aponta que **o plano mais premium (Ultimate, na análise) tem maior potencial de receita por usuário**. 
* Por isso, **optei pelo modelo otimizado pelo score f1**, pois é uma métrica que otimiza o score para a classe Ultra (1), reduzindo erros nas previsões para essa classe.
    * O melhor conjunto de hiperparâmetros encontrado para otimização pelo F1 foi:
        * ``class_weight='balanced'``
        * ``criterion='entropy'``
        * ``max_depth=10``
        * ``n_estimators=200``
    * No conjunto de validação, esse modelo alcançou **F1-score de 0,63** para a classe positiva, além de manter uma **acurácia de aproximadamente 80%**.
* No entando, mesmo com a otimização, o recall se manteve bastante próximo a um lançamento de moeda (*coin toss*), com valores próximos a 50% de erros e acertos. O que indica um viés no modelo, herdado do desbalanço entre as classes.
    

## Testando o modelo escolhido

In [151]:
predictions = chosen_model.predict(features_test)
chosen_accuracy = accuracy_score(target_test, predictions)
print(f"Acurácia do modelo escolhido no conjunto de teste: {chosen_accuracy:.2f}")
print("\nRelatório de Classificação no conjunto de teste:\n", classification_report(target_test, predictions))
print("\nMatriz de Confusão no conjunto de teste:\n", confusion_matrix(target_test, predictions))

Acurácia do modelo escolhido no conjunto de teste: 0.81

Relatório de Classificação no conjunto de teste:
               precision    recall  f1-score   support

           0       0.83      0.91      0.87       448
           1       0.74      0.58      0.65       195

    accuracy                           0.81       643
   macro avg       0.79      0.75      0.76       643
weighted avg       0.81      0.81      0.80       643


Matriz de Confusão no conjunto de teste:
 [[409  39]
 [ 82 113]]


O desempenho obtido no conjunto de teste confirma as expectativas geradas durante a validação. O modelo escolhido alcançou **81% de acurácia**, com **precisão de 0,74**, **recall de 0,58** e **F1-score de 0,65** para a classe *Ultra*. Esses resultados são muito próximos dos observados no conjunto de validação, indicando boa capacidade de generalização e ausência de sinais relevantes de sobreajuste. 

## Fazendo um teste comparativo com DummyClassifier

In [ ]:
# Importando DummyClassifier para comparação
from sklearn.dummy import DummyClassifier
dummy_model = DummyClassifier(strategy='most_frequent', random_state=42) # estratégia 'most_frequent' para sempre prever a classe mais frequente
dummy_model.fit(features_train, target_train)

,strategy,'prior'
,random_state,42
,constant,None


In [147]:
# Comparando previsões do modelo escolhido com o DummyClassifier
dummy_predictions = dummy_model.predict(features_test)
dummy_accuracy = accuracy_score(target_test, dummy_predictions)
print(f"Acurácia do DummyClassifier no conjunto de teste: {dummy_accuracy:.2f}")
print("\nRelatório de Classificação do DummyClassifier no conjunto de teste:\n", classification_report(target_test, dummy_predictions))
print("\nMatriz de Confusão do DummyClassifier no conjunto de teste:\n", confusion_matrix(target_test, dummy_predictions)) 

Acurácia do DummyClassifier no conjunto de teste: 0.70

Relatório de Classificação do DummyClassifier no conjunto de teste:
               precision    recall  f1-score   support

           0       0.70      1.00      0.82       448
           1       0.00      0.00      0.00       195

    accuracy                           0.70       643
   macro avg       0.35      0.50      0.41       643
weighted avg       0.49      0.70      0.57       643


Matriz de Confusão do DummyClassifier no conjunto de teste:
 [[448   0]
 [195   0]]


e:\DataScience\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\DataScience\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
e:\DataScience\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [149]:
# Criando um dummy classifier para comparação, que prevê igualmente as duas classes
coin_model = DummyClassifier(strategy='uniform', random_state=42) # estratégia 'uniform' para prever aleatoriamente as classes com igual probabilidade
coin_model.fit(features_train, target_train)


,strategy,'uniform'
,random_state,42
,constant,None


In [150]:
# Comparando previsões do modelo escolhido com o Coin Model
coin_predictions = coin_model.predict(features_test)
coin_accuracy = accuracy_score(target_test, coin_predictions)
print(f"Acurácia do Coin Model no conjunto de teste: {coin_accuracy:.2f}")
print("\nRelatório de Classificação do Coin Model no conjunto de teste:\n", classification_report(target_test, coin_predictions))
print("\nMatriz de Confusão do Coin Model no conjunto de teste:\n", confusion_matrix(target_test, coin_predictions)) 

Acurácia do Coin Model no conjunto de teste: 0.51

Relatório de Classificação do Coin Model no conjunto de teste:
               precision    recall  f1-score   support

           0       0.72      0.49      0.58       448
           1       0.32      0.55      0.41       195

    accuracy                           0.51       643
   macro avg       0.52      0.52      0.49       643
weighted avg       0.60      0.51      0.53       643


Matriz de Confusão do Coin Model no conjunto de teste:
 [[220 228]
 [ 87 108]]


In [153]:
# Sanity check - verificando se o modelo escolhido tem desempenho melhor do que os modelos dummy, o que indicaria que ele aprendeu algo útil com os dados e não está apenas fazendo previsões aleatórias ou sempre prevendo a classe majoritária.
if chosen_accuracy > dummy_accuracy and chosen_accuracy > coin_accuracy:
    print("O modelo escolhido tem desempenho melhor do que os modelos dummy")
else:
    print("O modelo escolhido não tem desempenho melhor do que os modelos dummy, o que pode indicar que ele não aprendeu algo útil com os dados ou que o conjunto de teste é muito pequeno para avaliar o desempenho do modelo de forma confiável.")

O modelo escolhido tem desempenho melhor do que os modelos dummy


In [154]:
# Checando para f1-score, que é a métrica que priorizamos para escolher o modelo final, para ver se o modelo escolhido tem desempenho melhor do que os modelos dummy também nessa métrica, o que reforçaria a conclusão de que ele aprendeu algo útil com os dados.
from sklearn.metrics import f1_score
f1_chosen = f1_score(target_test, predictions)
f1_dummy = f1_score(target_test, dummy_predictions)
f1_coin = f1_score(target_test, coin_predictions)
print(f"F1-score do modelo escolhido no conjunto de teste: {f1_chosen:.2f}")
print(f"F1-score do DummyClassifier no conjunto de teste: {f1_dummy:.2f}")
print(f"F1-score do Coin Model no conjunto de teste: {f1_coin:.2f}")

F1-score do modelo escolhido no conjunto de teste: 0.65
F1-score do DummyClassifier no conjunto de teste: 0.00
F1-score do Coin Model no conjunto de teste: 0.41


In [156]:
if f1_chosen > f1_dummy and f1_chosen > f1_coin:
    print("O modelo escolhido tem f1-score melhor do que os modelos dummy")
else:
    print("O modelo escolhido não tem f1-score melhor do que os modelos dummy, o que pode indicar que ele não aprendeu algo útil com os dados ou que o conjunto de teste é muito pequeno para avaliar o desempenho do modelo de forma confiável.")

O modelo escolhido tem f1-score melhor do que os modelos dummy


## Conclusão do Sanity Check:

* O modelo performa melhor que simplesmente escolher a classe mais frequente ou lançar uma moeda. O que indica que de fato, o modelo aprendeu algo com os dados e pode gerar previsões úteis, apesar do viés observado.

--------

## Salvando o modelo num arquivo joblib

In [158]:
# Salvando o modelo escolhido para uso futuro
import joblib

joblib.dump(chosen_model, 'models/best_model.joblib')

['models/best_model.joblib']

In [165]:
# Testando o modelo salvo para garantir que ele foi salvo corretamente e pode ser carregado e usado para fazer previsões
loaded_model = joblib.load('models/best_model.joblib')
display(loaded_model)

,n_estimators,200
,criterion,'entropy'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [166]:
predictions_loaded = loaded_model.predict(features_test)
loaded_accuracy = accuracy_score(target_test, predictions_loaded)
print(f"Acurácia do modelo carregado no conjunto de teste: {loaded_accuracy:.2f}")

Acurácia do modelo carregado no conjunto de teste: 0.81
